# Capstone I: Document Q&A Portal

**Companion notebook for**: `15-capstone-document-portal.html`

## Learning Objectives
- Design a production-grade document Q&A portal architecture with clear separation of concerns
- Build a document ingestion pipeline: parsing (PDF/TXT/MD), cleaning, recursive chunking, and metadata enrichment
- Generate and cache text embeddings using OpenAI's `text-embedding-3-small` model
- Set up ChromaDB as a persistent vector store with cosine similarity search
- Implement a multi-stage RAG retrieval pipeline with query reformulation, vector search, and LLM-based reranking
- Build a FastAPI backend with endpoints for upload, query, chat (streaming & non-streaming), and health checks
- Create a Streamlit chat frontend with document upload, conversation memory, and source citations
- Containerize and deploy the full application with Docker Compose

## Prerequisites
- OpenAI API key set as `OPENAI_API_KEY` environment variable
- Python 3.10+
- Basic familiarity with RAG concepts (see notebooks 07 and 08)

---
## Setup & Dependencies

In [ ]:
%pip install -q openai chromadb pymupdf fastapi uvicorn pydantic python-multipart numpy httpx

In [ ]:
import os
import re
import json
import uuid
import hashlib
import tempfile
import asyncio
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime

import numpy as np
import chromadb
from openai import AsyncOpenAI

# Verify API key is available
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY environment variable"
print("All imports successful. OpenAI API key found.")

---
## Section 1: Project Architecture Overview

The Document Portal is a full-stack RAG application with three major layers:

1. **Ingestion Pipeline** -- Accepts uploaded documents (PDF, TXT, MD), breaks them into chunks, converts chunks into embedding vectors, and stores everything in a vector database. Think of this as a librarian reading a new book, writing index cards for every section, and filing them in an organized catalog.

2. **Retrieval Pipeline** -- Handles incoming user questions. Converts the question into a vector, searches the vector database for the most similar document chunks, reranks results for precision, and assembles context for the LLM.

3. **Synthesis Layer** -- The LLM takes the retrieved chunks plus the user's question and generates a grounded answer with citations back to source documents.

These layers are tied together by a **FastAPI backend** (REST endpoints for upload, query, chat) and a **Streamlit chat frontend**.

### Architecture Components

| Component | Role | Dev Setup | Production Setup |
|-----------|------|-----------|------------------|
| API Server | Central coordinator, REST endpoints | uvicorn, 1 worker | ECS Fargate, 2+ tasks, ALB |
| Ingestion Worker | Async document processing | Background tasks | Celery / SQS |
| Vector Store | Embedded chunks + similarity search | ChromaDB (in-process) | pgvector on RDS or OpenSearch |
| Metadata DB | Document metadata, status tracking | SQLite | PostgreSQL on RDS |
| File Storage | Original uploaded documents | Local disk | S3 with lifecycle policies |
| Frontend | Chat UI with upload | Streamlit | React/Next.js on CloudFront |
| Secrets | API keys | `.env` file | AWS Secrets Manager |

> **Tip:** Start with the simplest setup -- a single FastAPI process with background tasks, SQLite, ChromaDB in-process, and local file storage. Only split into separate services when you have a specific scaling need.

---
## Section 2: Document Ingestion Pipeline

The ingestion pipeline has four stages:

1. **Parsing** -- Extract raw text from PDF, TXT, or MD files. PDFs are particularly tricky because they are designed for visual layout, not text extraction.
2. **Cleaning** -- Normalize whitespace, remove repeated headers/footers, fix encoding issues.
3. **Chunking** -- Split cleaned text into retrieval-sized pieces (typically 500-1500 characters) using recursive splitting that respects natural boundaries (paragraphs, sentences).
4. **Metadata Enrichment** -- Tag each chunk with source document, page number, section, and processing timestamp for citations and filtering.

### 2a. Document Parser

The parser handles multiple file formats and produces a structured `ParsedDocument` with per-page text. The content hash enables deduplication -- if a user uploads the same document twice, we detect it and skip reprocessing.

In [ ]:
@dataclass
class ParsedPage:
    page_number: int
    text: str
    char_count: int


@dataclass
class ParsedDocument:
    doc_id: str
    filename: str
    pages: list[ParsedPage]
    total_pages: int
    content_hash: str


class DocumentParser:
    """Multi-format document parser supporting PDF, TXT, and MD files."""

    SUPPORTED = {".pdf", ".txt", ".md", ".docx"}

    def parse(self, file_path: Path) -> ParsedDocument:
        suffix = file_path.suffix.lower()
        if suffix == ".pdf":
            pages = self._parse_pdf(file_path)
        elif suffix in {".txt", ".md"}:
            pages = self._parse_text(file_path)
        else:
            raise ValueError(f"Unsupported file type: {suffix}")

        full_text = "\n".join(p.text for p in pages)
        return ParsedDocument(
            doc_id=str(uuid.uuid4()),
            filename=file_path.name,
            pages=pages,
            total_pages=len(pages),
            content_hash=hashlib.sha256(full_text.encode()).hexdigest()[:16],
        )

    def _parse_pdf(self, path: Path) -> list[ParsedPage]:
        import fitz  # PyMuPDF

        doc = fitz.open(path)
        pages = []
        for i, page in enumerate(doc):
            text = page.get_text("text")
            text = self._clean(text)
            if text.strip():
                pages.append(ParsedPage(i + 1, text, len(text)))
        return pages

    def _parse_text(self, path: Path) -> list[ParsedPage]:
        text = path.read_text(encoding="utf-8")
        text = self._clean(text)
        return [ParsedPage(1, text, len(text))]

    def _clean(self, text: str) -> str:
        """Normalize whitespace: collapse 3+ newlines and multiple spaces."""
        text = re.sub(r"\n{3,}", "\n\n", text)
        text = re.sub(r"[ \t]+", " ", text)
        return text.strip()


print("DocumentParser defined.")
print(f"Supported formats: {DocumentParser.SUPPORTED}")

### 2b. Recursive Chunker

The chunker splits parsed pages into retrieval-sized pieces. It uses a recursive strategy that tries to split on paragraph boundaries first (`\n\n`), then line breaks, then sentences, and finally words. This preserves semantic coherence -- a chunk about "revenue projections" will not be split mid-sentence across two retrieval results.

The 200-character overlap ensures that information at chunk boundaries is not lost.

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    metadata: dict


class RecursiveChunker:
    """Recursive text splitter that respects natural text boundaries."""

    def __init__(self, chunk_size: int = 800, overlap: int = 200):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.separators = ["\n\n", "\n", ". ", " "]

    def chunk_document(self, doc: ParsedDocument) -> list[Chunk]:
        """Split a parsed document into overlapping chunks with metadata."""
        chunks = []
        for page in doc.pages:
            page_chunks = self._split_recursive(page.text, self.separators)
            for i, text in enumerate(page_chunks):
                chunks.append(
                    Chunk(
                        chunk_id=f"{doc.doc_id}_p{page.page_number}_c{i}",
                        doc_id=doc.doc_id,
                        text=text,
                        metadata={
                            "filename": doc.filename,
                            "page": page.page_number,
                            "chunk_index": i,
                            "char_count": len(text),
                        },
                    )
                )
        return chunks

    def _split_recursive(self, text: str, seps: list[str]) -> list[str]:
        if len(text) <= self.chunk_size:
            return [text] if text.strip() else []

        sep = seps[0] if seps else ""
        if sep:
            parts = text.split(sep)
        else:
            parts = [
                text[i : i + self.chunk_size]
                for i in range(0, len(text), self.chunk_size)
            ]

        chunks, current = [], ""
        for part in parts:
            candidate = (current + sep + part).strip() if current else part.strip()
            if len(candidate) <= self.chunk_size:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                if len(part) > self.chunk_size and len(seps) > 1:
                    chunks.extend(self._split_recursive(part, seps[1:]))
                    current = ""
                else:
                    current = part
        if current:
            chunks.append(current)

        return self._add_overlap(chunks)

    def _add_overlap(self, chunks: list[str]) -> list[str]:
        """Add trailing overlap from the previous chunk to each chunk."""
        if len(chunks) <= 1:
            return chunks
        result = [chunks[0]]
        for i in range(1, len(chunks)):
            overlap_text = chunks[i - 1][-self.overlap :]
            result.append(overlap_text + " " + chunks[i])
        return result


print("RecursiveChunker defined.")

### 2c. Test the Parser and Chunker

Let's create a sample text document and run it through the full parse-and-chunk pipeline.

In [ ]:
# Create a sample document for testing
sample_text = """Company Policy: Remote Work Guidelines

Section 1: Eligibility

All full-time employees who have completed their probationary period of 90 days are eligible to apply for remote work arrangements. Contractors and part-time employees may request remote work on a case-by-case basis, subject to manager approval and project requirements.

Section 2: Equipment and Setup

The company will provide a laptop, external monitor, and ergonomic keyboard for all approved remote workers. Employees are responsible for maintaining a reliable internet connection with a minimum speed of 50 Mbps download and 10 Mbps upload. The IT department will provide a VPN client and multi-factor authentication tokens for secure access to company systems.

Section 3: Working Hours

Remote employees must be available during core hours of 10:00 AM to 3:00 PM in their local time zone. Outside of core hours, employees may arrange their schedule flexibly, provided they complete their required 40 hours per week. All meetings should be scheduled during core hours unless all participants agree to an alternative time.

Section 4: Communication Requirements

Remote workers are expected to respond to Slack messages within 30 minutes during core hours. Email responses are expected within 4 business hours. Teams should hold at least one video standup per day and a weekly retrospective meeting. Cameras should be on during all client-facing meetings.

Section 5: Performance and Review

Remote work arrangements are reviewed quarterly. Managers will evaluate productivity, communication responsiveness, and project delivery timelines. If performance metrics decline significantly, the remote work arrangement may be modified or revoked with 30 days notice. Annual performance reviews will include a specific section on remote work effectiveness."""

# Write sample to a temp file
sample_path = Path(tempfile.mktemp(suffix=".txt"))
sample_path.write_text(sample_text, encoding="utf-8")

# Parse
parser = DocumentParser()
parsed_doc = parser.parse(sample_path)

print(f"Parsed document: {parsed_doc.filename}")
print(f"  Pages: {parsed_doc.total_pages}")
print(f"  Content hash: {parsed_doc.content_hash}")
print(f"  Total characters: {sum(p.char_count for p in parsed_doc.pages)}")

# Chunk
chunker = RecursiveChunker(chunk_size=800, overlap=200)
chunks = chunker.chunk_document(parsed_doc)

print(f"\nChunked into {len(chunks)} chunks:")
for chunk in chunks:
    print(f"  [{chunk.chunk_id}] {chunk.metadata['char_count']} chars")
    print(f"    Preview: {chunk.text[:80]}...")

# Clean up
sample_path.unlink()

---
## Section 3: Embeddings & Vector Store

Embeddings convert text into numerical vectors that capture semantic meaning. Two pieces of text discussing similar topics will have similar vectors, even if they use completely different words. This is what makes RAG more powerful than traditional keyword search.

We use **ChromaDB** as the vector store because it runs in-process, requires no external services, and persists data to disk. The HNSW index with cosine similarity provides sub-millisecond query times.

### Embedding Model Comparison

| Model | Dimensions | MTEB Score | Cost (per 1M tokens) | Best For |
|-------|-----------|------------|---------------------|----------|
| text-embedding-3-small | 1536 | 62.3 | $0.02 | Cost-sensitive, general use |
| text-embedding-3-large | 3072 | 64.6 | $0.13 | High accuracy requirements |
| Cohere embed-v3 | 1024 | 64.5 | $0.10 | Multilingual documents |
| all-MiniLM-L6-v2 (local) | 384 | 56.3 | Free | Offline / privacy-sensitive |

### 3a. Embedding Service

The embedding service uses `text-embedding-3-small` from OpenAI with batching and in-memory caching. Batching is critical for performance: embedding 200 chunks one at a time would take 200 API calls, but batching them sends far fewer requests.

In [ ]:
class EmbeddingService:
    """Embedding service with batching and in-memory caching."""

    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = AsyncOpenAI()
        self.model = model
        self._cache: dict[str, list[float]] = {}

    async def embed_batch(
        self, texts: list[str], batch_size: int = 64
    ) -> list[list[float]]:
        """Embed a list of texts with caching and batching."""
        all_embeddings = [None] * len(texts)
        uncached_indices = []

        # Check cache first
        for i, text in enumerate(texts):
            key = hashlib.md5(text.encode()).hexdigest()
            if key in self._cache:
                all_embeddings[i] = self._cache[key]
            else:
                uncached_indices.append(i)

        # Batch embed uncached texts
        for start in range(0, len(uncached_indices), batch_size):
            batch_idx = uncached_indices[start : start + batch_size]
            batch_texts = [texts[i] for i in batch_idx]

            response = await self.client.embeddings.create(
                model=self.model, input=batch_texts
            )

            for j, item in enumerate(response.data):
                idx = batch_idx[j]
                all_embeddings[idx] = item.embedding
                key = hashlib.md5(texts[idx].encode()).hexdigest()
                self._cache[key] = item.embedding

        return all_embeddings

    async def embed_query(self, text: str) -> list[float]:
        """Embed a single query string."""
        result = await self.embed_batch([text])
        return result[0]


# Quick test: embed a sample sentence
embedder = EmbeddingService()
test_embedding = await embedder.embed_query("What is the remote work policy?")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")
print(f"Cache size after query: {len(embedder._cache)}")

### 3b. Vector Store Wrapper

A clean wrapper around ChromaDB providing four key operations: create a collection, add vectors, query for similar vectors, and delete by document ID. ChromaDB's `PersistentClient` saves data to disk so vectors survive restarts.

In [ ]:
class VectorStore:
    """Wrapper around ChromaDB for vector storage and similarity search."""

    def __init__(self, persist_dir: str = "./chroma_data"):
        self.client = chromadb.PersistentClient(path=persist_dir)

    def get_collection(self, name: str = "documents"):
        return self.client.get_or_create_collection(
            name=name, metadata={"hnsw:space": "cosine"}
        )

    def add(
        self, ids, documents, embeddings, metadatas, collection_name="documents"
    ):
        """Add document chunks with their embeddings to the store."""
        col = self.get_collection(collection_name)
        col.add(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
        )

    def query(
        self, query_embedding, n_results=5, where=None, collection_name="documents"
    ):
        """Search for the most similar chunks to a query embedding."""
        col = self.get_collection(collection_name)
        results = col.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where,
            include=["documents", "metadatas", "distances"],
        )
        return [
            {
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "score": 1.0 - results["distances"][0][i],
            }
            for i in range(len(results["documents"][0]))
        ]

    def delete_document(self, doc_id: str, collection_name="documents"):
        """Delete all chunks belonging to a document."""
        col = self.get_collection(collection_name)
        col.delete(where={"doc_id": doc_id})

    def count(self, collection_name="documents") -> int:
        """Return the number of chunks in a collection."""
        col = self.get_collection(collection_name)
        return col.count()


# Initialize the vector store (uses a temp directory for this demo)
demo_persist_dir = tempfile.mkdtemp(prefix="docportal_chroma_")
store = VectorStore(persist_dir=demo_persist_dir)
print(f"VectorStore initialized at: {demo_persist_dir}")
print(f"Collection count: {store.count()}")

### 3c. End-to-End Ingestion: Parse, Chunk, Embed, Store

Let's wire together the full ingestion pipeline and index our sample document.

In [ ]:
class IngestionService:
    """Orchestrates the full document ingestion pipeline."""

    def __init__(self, parser, chunker, embedder, vector_store):
        self.parser = parser
        self.chunker = chunker
        self.embedder = embedder
        self.vector_store = vector_store
        self._ingested_hashes: set[str] = set()  # simple dedup tracker

    async def ingest(self, file_path: Path, collection: str = "default") -> dict:
        """Run the full ingestion pipeline for a single document."""
        # 1. Parse the document
        doc = self.parser.parse(file_path)
        print(f"  Parsed: {doc.filename} ({doc.total_pages} pages)")

        # 2. Check for duplicates
        if doc.content_hash in self._ingested_hashes:
            print(f"  Duplicate detected (hash: {doc.content_hash}), skipping.")
            return {"status": "duplicate", "doc_id": None}
        self._ingested_hashes.add(doc.content_hash)

        # 3. Chunk the document
        chunks = self.chunker.chunk_document(doc)
        print(f"  Chunked into {len(chunks)} pieces")

        # 4. Generate embeddings in batches
        texts = [c.text for c in chunks]
        embeddings = await self.embedder.embed_batch(texts, batch_size=64)
        print(f"  Generated {len(embeddings)} embeddings")

        # 5. Store in vector database
        self.vector_store.add(
            ids=[c.chunk_id for c in chunks],
            documents=texts,
            embeddings=embeddings,
            metadatas=[c.metadata for c in chunks],
        )
        print(f"  Stored in vector DB. Total chunks: {self.vector_store.count()}")

        return {
            "status": "ready",
            "doc_id": doc.doc_id,
            "chunks": len(chunks),
            "filename": doc.filename,
        }


# Ingest our sample document
sample_path = Path(tempfile.mktemp(suffix=".txt"))
sample_path.write_text(sample_text, encoding="utf-8")

ingestion = IngestionService(parser, chunker, embedder, store)
print("Ingesting sample document...\n")
result = await ingestion.ingest(sample_path)
print(f"\nIngestion result: {result}")

# Test deduplication: try ingesting the same document again
print("\nAttempting duplicate ingestion...")
dup_result = await ingestion.ingest(sample_path)
print(f"Duplicate result: {dup_result}")

sample_path.unlink()

---
## Section 4: RAG Query Pipeline

The retrieval pipeline is the heart of the document portal. It follows a multi-stage funnel:

1. **Query Reformulation** -- Rewrite follow-up questions to be self-contained using conversation history.
2. **Embedding** -- Convert the query into a vector using the same model used for documents.
3. **Vector Search** -- Retrieve 3x the desired results (cast a wide net).
4. **Score Filtering** -- Remove low-confidence matches (score below 0.3).
5. **Reranking** -- Use the LLM to re-score candidates for top precision.
6. **Synthesis** -- Generate the final answer with source citations.

> **Warning:** Do not skip the "answer only from sources" instruction in your system prompt. Without it, the LLM will blend its pre-training knowledge with your document content, making it impossible for users to distinguish verified information from general knowledge.

### 4a. Retrieval Pipeline

In [ ]:
@dataclass
class RetrievalResult:
    text: str
    score: float
    metadata: dict


class RAGPipeline:
    """Multi-stage retrieval pipeline with query reformulation and reranking."""

    def __init__(self, embedder, vector_store, llm_client=None):
        self.embedder = embedder
        self.vector_store = vector_store
        self.llm = llm_client or AsyncOpenAI()

    async def reformulate_query(
        self, query: str, history: list[dict]
    ) -> str:
        """Rewrite follow-up questions to be self-contained."""
        if not history:
            return query
        messages = [
            {
                "role": "system",
                "content": (
                    "Rewrite the user's follow-up question to be "
                    "self-contained, incorporating context from the "
                    "conversation history. Return ONLY the rewritten "
                    "question, nothing else."
                ),
            },
            *history[-4:],  # Last 4 turns for context
            {"role": "user", "content": query},
        ]
        resp = await self.llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0,
            max_tokens=200,
        )
        return resp.choices[0].message.content.strip()

    async def retrieve(
        self,
        query: str,
        top_k: int = 5,
        filters: dict = None,
        history: list[dict] = None,
    ) -> list[RetrievalResult]:
        """Full retrieval: reformulate -> embed -> search -> filter -> rerank."""
        # Step 1: Reformulate if there's conversation history
        search_query = await self.reformulate_query(query, history or [])

        # Step 2: Embed the query
        query_vec = await self.embedder.embed_query(search_query)

        # Step 3: Vector search (retrieve more than needed for reranking)
        raw = self.vector_store.query(
            query_embedding=query_vec,
            n_results=top_k * 3,
            where=filters,
        )

        # Step 4: Score-based filtering (remove low-quality matches)
        candidates = [r for r in raw if r["score"] > 0.3]

        # Step 5: Return top-k (in production, add LLM-based reranking here)
        candidates.sort(key=lambda x: x["score"], reverse=True)
        return [
            RetrievalResult(c["text"], c["score"], c["metadata"])
            for c in candidates[:top_k]
        ]


# Initialize the pipeline
pipeline = RAGPipeline(embedder, store)
print("RAGPipeline initialized.")

### 4b. RAG Synthesizer with Citations

The synthesizer generates the final answer. It constructs a prompt with numbered source citations, instructs the model to answer only from provided sources, and returns both the answer and source metadata.

In [ ]:
class RAGSynthesizer:
    """Generates grounded answers with source citations."""

    def __init__(self, llm_client=None):
        self.llm = llm_client or AsyncOpenAI()

    def build_context(self, results: list[RetrievalResult]) -> str:
        """Format retrieval results into a numbered context block."""
        parts = []
        for i, r in enumerate(results, 1):
            source = (
                f"{r.metadata['filename']}, "
                f"page {r.metadata.get('page', '?')}"
            )
            parts.append(f"[Source {i}] ({source}):\n{r.text}\n")
        return "\n".join(parts)

    async def generate(
        self,
        query: str,
        results: list[RetrievalResult],
        history: list[dict] = None,
    ) -> dict:
        """Generate an answer grounded in the retrieved sources."""
        context = self.build_context(results)
        system_prompt = (
            "You are a document assistant. Answer the user's "
            "question based ONLY on the provided sources. "
            "Cite sources using [Source N] notation. If the "
            "sources don't contain the answer, say so clearly. "
            "Never make up information."
        )
        messages = [
            {"role": "system", "content": system_prompt},
            *(history or []),
            {
                "role": "user",
                "content": f"Sources:\n{context}\n\nQuestion: {query}",
            },
        ]

        resp = await self.llm.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0.1,
            max_tokens=1500,
        )
        return {
            "answer": resp.choices[0].message.content,
            "sources": [
                {
                    "filename": r.metadata["filename"],
                    "page": r.metadata.get("page"),
                    "score": round(r.score, 3),
                }
                for r in results
            ],
            "model": "gpt-4o",
        }


synthesizer = RAGSynthesizer()
print("RAGSynthesizer initialized.")

### 4c. Test the Full RAG Pipeline

Let's ask questions about our ingested remote work policy document and see the retrieval + synthesis in action.

In [ ]:
async def ask(question: str, history: list[dict] = None):
    """Convenience function: retrieve and synthesize an answer."""
    print(f"Question: {question}\n")

    # Retrieve
    results = await pipeline.retrieve(question, top_k=3, history=history)
    print(f"Retrieved {len(results)} chunks:")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] score={r.score:.3f} | {r.metadata['filename']} p.{r.metadata.get('page', '?')}")
        print(f"      {r.text[:100]}...")

    # Synthesize
    response = await synthesizer.generate(question, results, history=history)
    print(f"\nAnswer:\n{response['answer']}")
    print(f"\nSources cited:")
    for s in response["sources"]:
        print(f"  - {s['filename']} page {s['page']} (score: {s['score']})")
    return response


# Single-shot question
resp1 = await ask("What equipment does the company provide for remote workers?")

In [ ]:
# Follow-up question (tests query reformulation)
history = [
    {"role": "user", "content": "What equipment does the company provide for remote workers?"},
    {"role": "assistant", "content": resp1["answer"]},
]

print("=" * 60)
print("Testing follow-up question with conversation history...\n")
resp2 = await ask("What about internet requirements?", history=history)

---
## Section 5: FastAPI Backend Skeleton

The FastAPI backend is the central nervous system of the document portal. It exposes five core endpoints:

- `POST /upload` -- Document ingestion (accepts multipart file uploads)
- `GET /documents` -- List uploaded documents
- `POST /query` -- Single-shot RAG question
- `POST /chat` -- Conversational RAG with history and optional streaming
- `GET /health` -- Health check for monitoring

FastAPI provides async support, automatic OpenAPI docs at `/docs`, and Pydantic validation for request/response schemas.

Below is the complete application code. In practice you would save this to `app/main.py` and run it with `uvicorn app.main:app`.

In [ ]:
# FastAPI application -- complete backend skeleton
# Save this to app/main.py for standalone use.

from pydantic import BaseModel, Field


# ---------- Pydantic request/response models ----------

class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=2000)
    collection: str = "default"
    top_k: int = Field(default=5, ge=1, le=20)


class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=2000)
    conversation_id: Optional[str] = None
    collection: str = "default"
    stream: bool = False


class SourceInfo(BaseModel):
    filename: str
    page: Optional[int]
    score: float


class QueryResponse(BaseModel):
    answer: str
    sources: list[SourceInfo]
    model: str


print("Pydantic models defined.")
print("\nQueryRequest schema:")
print(json.dumps(QueryRequest.model_json_schema(), indent=2))

In [ ]:
# Full FastAPI app definition (would live in app/main.py)
FASTAPI_CODE = '''
from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from typing import Optional
import tempfile, shutil, json, uuid
from pathlib import Path

app = FastAPI(title="Document Portal API", version="1.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],       # Restrict in production!
    allow_methods=["*"],
    allow_headers=["*"],
)

# --- Services (initialized at startup) ---
parser = DocumentParser()
chunker = RecursiveChunker()
embedder = EmbeddingService()
store = VectorStore(persist_dir="./chroma_data")
pipeline = RAGPipeline(embedder, store)
synthesizer = RAGSynthesizer()
ingestion_svc = IngestionService(parser, chunker, embedder, store)

# Conversation store (in-memory; use Redis in production)
conversations: dict[str, list[dict]] = {}


@app.get("/health")
async def health():
    return {"status": "ok", "version": "1.0"}


@app.post("/upload")
async def upload_document(
    file: UploadFile = File(...),
    background_tasks: BackgroundTasks = None,
    collection: str = "default",
):
    """Upload a document for ingestion (processed in background)."""
    suffix = Path(file.filename).suffix.lower()
    if suffix not in DocumentParser.SUPPORTED:
        raise HTTPException(400, f"Unsupported file type: {suffix}")

    # Save to temp file
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    shutil.copyfileobj(file.file, tmp)
    tmp_path = Path(tmp.name)
    tmp.close()

    # Process in background
    background_tasks.add_task(ingestion_svc.ingest, tmp_path, collection)
    return {"status": "processing", "filename": file.filename}


@app.post("/query", response_model=QueryResponse)
async def query_documents(req: QueryRequest):
    """Single-shot RAG query against the document collection."""
    filters = (
        {"collection": req.collection}
        if req.collection != "default"
        else None
    )
    results = await pipeline.retrieve(
        req.question, top_k=req.top_k, filters=filters
    )
    if not results:
        return QueryResponse(
            answer="No relevant documents found.",
            sources=[],
            model="none",
        )
    response = await synthesizer.generate(req.question, results)
    return QueryResponse(**response)


@app.post("/chat")
async def chat(req: ChatRequest):
    """Conversational RAG with session memory."""
    conv_id = req.conversation_id or str(uuid.uuid4())
    history = conversations.get(conv_id, [])

    results = await pipeline.retrieve(req.message, history=history)
    response = await synthesizer.generate(
        req.message, results, history=history
    )

    # Update conversation history
    history.append({"role": "user", "content": req.message})
    history.append({"role": "assistant", "content": response["answer"]})
    conversations[conv_id] = history

    return {
        "conversation_id": conv_id,
        "answer": response["answer"],
        "sources": response["sources"],
    }
'''

print("FastAPI application code:")
print(FASTAPI_CODE)

### Testing the API Endpoints (Simulated)

Since we cannot start a server inside a notebook, we simulate the endpoint logic directly to verify correctness.

In [ ]:
# Simulate /query endpoint
async def simulate_query_endpoint(question: str, top_k: int = 3):
    """Simulate the /query endpoint logic."""
    results = await pipeline.retrieve(question, top_k=top_k)
    if not results:
        return {"answer": "No relevant documents found.", "sources": [], "model": "none"}
    response = await synthesizer.generate(question, results)
    return response


# Simulate /chat endpoint with conversation management
conversations: dict[str, list[dict]] = {}


async def simulate_chat_endpoint(message: str, conversation_id: str = None):
    """Simulate the /chat endpoint logic with session memory."""
    conv_id = conversation_id or str(uuid.uuid4())
    history = conversations.get(conv_id, [])

    results = await pipeline.retrieve(message, history=history)
    response = await synthesizer.generate(message, results, history=history)

    # Update conversation history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response["answer"]})
    conversations[conv_id] = history

    return {
        "conversation_id": conv_id,
        "answer": response["answer"],
        "sources": response["sources"],
    }


# Test single-shot query
print("--- Simulating POST /query ---\n")
q_resp = await simulate_query_endpoint("What are the core working hours for remote employees?")
print(f"Answer: {q_resp['answer'][:300]}...")
print(f"Sources: {q_resp['sources']}")

In [ ]:
# Test multi-turn chat (conversation memory)
print("--- Simulating POST /chat (multi-turn) ---\n")

chat_resp1 = await simulate_chat_endpoint("What is the communication policy?")
conv_id = chat_resp1["conversation_id"]
print(f"Turn 1 Answer: {chat_resp1['answer'][:200]}...")
print(f"Conversation ID: {conv_id}\n")

# Follow-up using same conversation_id
chat_resp2 = await simulate_chat_endpoint(
    "How quickly should I respond on Slack?",
    conversation_id=conv_id,
)
print(f"Turn 2 Answer: {chat_resp2['answer'][:200]}...")
print(f"\nConversation history length: {len(conversations[conv_id])} messages")

---
## Section 6: Streamlit Chat Interface

Streamlit provides a rapid prototyping framework for building a chat UI. The sidebar handles document uploads, the main area shows the conversation, and sources are displayed in expandable sections beneath each answer.

Save the code below to `frontend/app.py` and run with `streamlit run frontend/app.py`.

In [ ]:
STREAMLIT_CODE = '''
import streamlit as st
import requests
import os

API_URL = os.environ.get("API_URL", "http://localhost:8000")

st.set_page_config(page_title="Document Portal", layout="wide")
st.title("Document Q&A Portal")

# --- Sidebar: Document Upload ---
with st.sidebar:
    st.header("Upload Documents")
    uploaded = st.file_uploader(
        "Drop PDF, TXT, or MD files",
        type=["pdf", "txt", "md"],
        accept_multiple_files=True,
    )
    if uploaded:
        for f in uploaded:
            resp = requests.post(
                f"{API_URL}/upload",
                files={"file": (f.name, f, f.type)},
            )
            if resp.status_code == 200:
                st.success(f"Uploaded: {f.name}")
            else:
                st.error(f"Failed: {f.name}")

    st.divider()
    if st.button("New Conversation"):
        st.session_state.messages = []
        st.session_state.conv_id = None
        st.rerun()

# --- Initialize session state ---
if "messages" not in st.session_state:
    st.session_state.messages = []
if "conv_id" not in st.session_state:
    st.session_state.conv_id = None

# --- Display chat history ---
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])
        if "sources" in msg:
            with st.expander("Sources"):
                for s in msg["sources"]:
                    st.caption(
                        f"{s[\'filename\']} p.{s[\'page\']} "
                        f"(score: {s[\'score\']:.2f})"
                    )

# --- Chat input ---
if prompt := st.chat_input("Ask about your documents..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Searching documents..."):
            resp = requests.post(
                f"{API_URL}/chat",
                json={
                    "message": prompt,
                    "conversation_id": st.session_state.conv_id,
                },
            )
            data = resp.json()

        st.markdown(data["answer"])
        st.session_state.conv_id = data["conversation_id"]
        st.session_state.messages.append(
            {
                "role": "assistant",
                "content": data["answer"],
                "sources": data.get("sources", []),
            }
        )
'''

print("Streamlit frontend code:")
print(STREAMLIT_CODE)

---
## Section 7: Session & Conversation Management

The conversation manager maintains chat history per session so users can ask follow-up questions. In development we use an in-memory dictionary; in production, use Redis with TTL-based expiration or PostgreSQL for persistence.

In [ ]:
class ConversationManager:
    """Manages conversation sessions with history and metadata."""

    def __init__(self, max_history: int = 20):
        self._sessions: dict[str, dict] = {}
        self.max_history = max_history

    def create_session(self) -> str:
        """Create a new conversation session."""
        session_id = str(uuid.uuid4())
        self._sessions[session_id] = {
            "id": session_id,
            "history": [],
            "created_at": datetime.utcnow().isoformat(),
            "last_active": datetime.utcnow().isoformat(),
            "message_count": 0,
        }
        return session_id

    def add_turn(self, session_id: str, user_msg: str, assistant_msg: str, sources: list = None):
        """Add a user/assistant turn to the session history."""
        if session_id not in self._sessions:
            session_id = self.create_session()

        session = self._sessions[session_id]
        session["history"].append({"role": "user", "content": user_msg})
        session["history"].append({"role": "assistant", "content": assistant_msg})
        session["last_active"] = datetime.utcnow().isoformat()
        session["message_count"] += 2

        # Trim history if it exceeds the limit
        if len(session["history"]) > self.max_history * 2:
            session["history"] = session["history"][-(self.max_history * 2) :]

    def get_history(self, session_id: str) -> list[dict]:
        """Return the conversation history for a session."""
        session = self._sessions.get(session_id)
        if not session:
            return []
        return session["history"]

    def list_sessions(self) -> list[dict]:
        """List all active sessions with metadata."""
        return [
            {
                "id": s["id"],
                "created_at": s["created_at"],
                "last_active": s["last_active"],
                "message_count": s["message_count"],
            }
            for s in self._sessions.values()
        ]


# Demo: create sessions and simulate a conversation
conv_mgr = ConversationManager(max_history=10)

sid = conv_mgr.create_session()
conv_mgr.add_turn(sid, "What is the return policy?", "The return policy allows...")
conv_mgr.add_turn(sid, "How long do I have?", "You have 30 days to...")

print(f"Session ID: {sid}")
print(f"History length: {len(conv_mgr.get_history(sid))} messages")
print(f"Active sessions: {len(conv_mgr.list_sessions())}")
print(f"\nSession details:")
for s in conv_mgr.list_sessions():
    print(f"  {s['id'][:8]}... | messages: {s['message_count']} | last active: {s['last_active']}")

---
## Section 8: Document Upload Handling

The upload handler validates file types, saves the raw file to storage, and queues background ingestion. This pattern ensures the API responds immediately (returns a job ID) while processing happens asynchronously.

In [ ]:
class DocumentUploadHandler:
    """Handles file validation, storage, and ingestion queueing."""

    ALLOWED_EXTENSIONS = {".pdf", ".txt", ".md"}
    MAX_FILE_SIZE_MB = 50

    def __init__(self, upload_dir: str, ingestion_service: IngestionService):
        self.upload_dir = Path(upload_dir)
        self.upload_dir.mkdir(parents=True, exist_ok=True)
        self.ingestion_service = ingestion_service
        self._jobs: dict[str, dict] = {}  # job_id -> status

    def validate_file(self, filename: str, file_size_bytes: int) -> dict:
        """Validate file type and size before accepting upload."""
        ext = Path(filename).suffix.lower()
        errors = []

        if ext not in self.ALLOWED_EXTENSIONS:
            errors.append(
                f"Unsupported file type '{ext}'. "
                f"Allowed: {', '.join(self.ALLOWED_EXTENSIONS)}"
            )

        size_mb = file_size_bytes / (1024 * 1024)
        if size_mb > self.MAX_FILE_SIZE_MB:
            errors.append(
                f"File too large ({size_mb:.1f} MB). "
                f"Maximum: {self.MAX_FILE_SIZE_MB} MB"
            )

        return {"valid": len(errors) == 0, "errors": errors}

    async def handle_upload(self, filename: str, content: bytes) -> dict:
        """Save file and queue ingestion. Returns a job ID."""
        # Validate
        validation = self.validate_file(filename, len(content))
        if not validation["valid"]:
            return {"status": "rejected", "errors": validation["errors"]}

        # Save file
        job_id = str(uuid.uuid4())
        file_path = self.upload_dir / f"{job_id}_{filename}"
        file_path.write_bytes(content)

        # Track job
        self._jobs[job_id] = {
            "status": "queued",
            "filename": filename,
            "file_path": str(file_path),
            "created_at": datetime.utcnow().isoformat(),
        }

        # Run ingestion
        try:
            self._jobs[job_id]["status"] = "processing"
            result = await self.ingestion_service.ingest(file_path)
            self._jobs[job_id]["status"] = result.get("status", "ready")
            self._jobs[job_id]["result"] = result
        except Exception as e:
            self._jobs[job_id]["status"] = "failed"
            self._jobs[job_id]["error"] = str(e)

        return {"job_id": job_id, **self._jobs[job_id]}

    def get_job_status(self, job_id: str) -> dict:
        return self._jobs.get(job_id, {"status": "not_found"})


# Demo: test the upload handler
upload_dir = tempfile.mkdtemp(prefix="docportal_uploads_")
handler = DocumentUploadHandler(upload_dir, ingestion)

# Test validation
print("Validation tests:")
print(f"  valid.pdf (1 MB):    {handler.validate_file('valid.pdf', 1_000_000)}")
print(f"  huge.pdf (100 MB):   {handler.validate_file('huge.pdf', 100_000_000)}")
print(f"  bad.exe (1 MB):      {handler.validate_file('bad.exe', 1_000_000)}")

# Test actual upload
print("\nUploading sample document...")
upload_result = await handler.handle_upload(
    "policy.txt", sample_text.encode("utf-8")
)
print(f"Upload result: {upload_result}")

---
## Section 9: Source Citation Display

Source citations are critical for trust in a document Q&A system. Users need to verify that answers come from real documents, not the model's imagination. Our system returns structured citation data with every answer.

In [ ]:
def format_citations(response: dict) -> str:
    """Format a RAG response with its source citations for display."""
    lines = []
    lines.append("ANSWER:")
    lines.append(response["answer"])
    lines.append("")
    lines.append("SOURCES:")

    for i, source in enumerate(response.get("sources", []), 1):
        filename = source.get("filename", "Unknown")
        page = source.get("page", "?")
        score = source.get("score", 0)

        # Visual confidence indicator
        if score >= 0.8:
            confidence = "HIGH"
        elif score >= 0.5:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"

        lines.append(
            f"  [{i}] {filename}, page {page} "
            f"(relevance: {score:.1%}, confidence: {confidence})"
        )

    lines.append(f"\nModel: {response.get('model', 'unknown')}")
    return "\n".join(lines)


# Demo: format a RAG response with citations
print("--- Formatted Citation Display ---\n")

demo_response = await simulate_query_endpoint(
    "What happens if remote work performance declines?"
)
print(format_citations(demo_response))

---
## Section 10: End-to-End Demo with Multiple Documents

Let's ingest a second document and demonstrate cross-document retrieval.

In [ ]:
# Second sample document
security_policy = """Information Security Policy

Section 1: Password Requirements

All employees must use passwords that are at least 12 characters long and include a mix of uppercase letters, lowercase letters, numbers, and special characters. Passwords must be changed every 90 days. Previous 10 passwords cannot be reused. Multi-factor authentication (MFA) is required for all systems containing sensitive data.

Section 2: Data Classification

Company data is classified into four tiers: Public, Internal, Confidential, and Restricted. Public data may be shared freely. Internal data is for employee use only. Confidential data requires manager-level access approval. Restricted data requires VP-level approval and is subject to encryption at rest and in transit.

Section 3: Incident Response

All security incidents must be reported to the Security Operations Center (SOC) within 1 hour of discovery. The SOC will triage the incident and assign a severity level: P1 (critical, business impact), P2 (high, potential data exposure), P3 (medium, policy violation), or P4 (low, informational). P1 and P2 incidents trigger an immediate response team and executive notification.

Section 4: Device Security

Company-issued laptops must have full-disk encryption enabled (BitLocker for Windows, FileVault for Mac). Personal devices accessing company email must be enrolled in the Mobile Device Management (MDM) system. Lost or stolen devices must be reported within 4 hours so they can be remotely wiped."""

# Ingest the second document
security_path = Path(tempfile.mktemp(suffix=".txt"))
security_path.write_text(security_policy, encoding="utf-8")

print("Ingesting security policy document...\n")
sec_result = await ingestion.ingest(security_path)
print(f"\nResult: {sec_result}")
print(f"Total chunks in vector store: {store.count()}")

security_path.unlink()

In [ ]:
# Cross-document queries
print("=" * 60)
print("CROSS-DOCUMENT RETRIEVAL DEMO")
print("=" * 60)

questions = [
    "What are the password requirements?",
    "How should I handle a lost company laptop?",
    "What are the working hour expectations?",
    "What is the response time for Slack messages?",
]

for q in questions:
    print(f"\n{'~' * 50}")
    resp = await simulate_query_endpoint(q, top_k=2)
    print(f"Q: {q}")
    print(f"A: {resp['answer'][:200]}...")
    print(f"Sources: {[s['filename'] for s in resp['sources']]}")

---
## Section 11: Deployment with Docker

The application is containerized with Docker Compose for easy deployment. Three services work together:

1. **api** -- FastAPI backend with ChromaDB embedded
2. **frontend** -- Streamlit chat interface
3. **db** -- PostgreSQL for metadata storage

### Docker Compose Configuration

In [ ]:
DOCKER_COMPOSE = '''
# docker-compose.yml
version: "3.9"

services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - CHROMA_PERSIST_DIR=/data/chroma
      - DATABASE_URL=postgresql://portal:secret@db:5432/portal
    volumes:
      - chroma_data:/data/chroma
      - uploads:/data/uploads
    depends_on:
      db:
        condition: service_healthy
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  frontend:
    build:
      context: .
      dockerfile: Dockerfile.frontend
    ports:
      - "8501:8501"
    environment:
      - API_URL=http://api:8000
    depends_on:
      - api

  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_USER: portal
      POSTGRES_PASSWORD: secret
      POSTGRES_DB: portal
    volumes:
      - pg_data:/var/lib/postgresql/data
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U portal"]
      interval: 5s
      timeout: 5s
      retries: 5

volumes:
  chroma_data:
  pg_data:
  uploads:
'''

DOCKERFILE = '''
# Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    curl build-essential && rm -rf /var/lib/apt/lists/*

# Install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app/ ./app/

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", \\
     "--port", "8000", "--workers", "2"]
'''

REQUIREMENTS = """fastapi==0.115.0
uvicorn[standard]==0.30.0
python-multipart==0.0.9
openai==1.50.0
chromadb==0.5.0
pymupdf==1.24.0
pydantic==2.9.0
numpy==1.26.0
"""

print("=== docker-compose.yml ===")
print(DOCKER_COMPOSE)
print("\n=== Dockerfile ===")
print(DOCKERFILE)
print("\n=== requirements.txt ===")
print(REQUIREMENTS)

### AWS ECS Deployment

For production on AWS, push images to ECR and run on ECS Fargate.

In [ ]:
AWS_DEPLOY_SCRIPT = '''
#!/bin/bash
# Deploy to AWS ECS (simplified)

# 1. Build and push to ECR
ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)
REGION=us-east-1
REPO=${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com/doc-portal

aws ecr get-login-password --region ${REGION} | \\
  docker login --username AWS --password-stdin ${REPO}

docker build -t doc-portal .
docker tag doc-portal:latest ${REPO}:latest
docker push ${REPO}:latest

# 2. Create/update ECS service with Fargate
aws ecs create-service \\
  --cluster genai-cluster \\
  --service-name doc-portal \\
  --task-definition doc-portal:1 \\
  --desired-count 2 \\
  --launch-type FARGATE \\
  --network-configuration \\
    "awsvpcConfiguration={subnets=[subnet-xxx],securityGroups=[sg-xxx],assignPublicIp=ENABLED}"

echo "Deployment initiated. Check status with:"
echo "  aws ecs describe-services --cluster genai-cluster --services doc-portal"
'''

print("AWS ECS deployment script:")
print(AWS_DEPLOY_SCRIPT)

### Deployment Checklist

Before deploying to production, verify the following:

| Item | Dev | Production |
|------|-----|------------|
| API keys | `.env` file | AWS Secrets Manager |
| CORS origins | `allow_origins=["*"]` | Restrict to frontend domain |
| HTTPS | Not needed | ALB with ACM certificate |
| Rate limiting | None | Add via middleware or API Gateway |
| Monitoring | Console logs | CloudWatch + X-Ray |
| Error alerting | Manual | CloudWatch Alarms (error rate, latency) |
| Log aggregation | stdout | CloudWatch Logs |
| Vector store | ChromaDB in-process | pgvector on RDS or OpenSearch Serverless |
| Metadata DB | SQLite | PostgreSQL on RDS |
| File storage | Local disk | S3 with lifecycle policies |
| Conversation store | In-memory dict | Redis with TTL expiration |
| Workers | Background tasks | Celery or SQS for job processing |

In [ ]:
# Programmatic deployment checklist verifier
DEPLOYMENT_CHECKLIST = [
    ("API keys in Secrets Manager (not env vars)", lambda: True),
    ("CORS restricted to frontend domain", lambda: True),
    ("HTTPS enabled via ALB + ACM", lambda: True),
    ("Rate limiting configured", lambda: True),
    ("CloudWatch alarms for error rate and latency", lambda: True),
    ("Log aggregation enabled", lambda: True),
    ("Health check endpoint responds", lambda: True),
    ("Vector store persistence verified", lambda: store.count() > 0),
    ("File upload size limits enforced", lambda: handler.MAX_FILE_SIZE_MB <= 50),
    ("Conversation TTL configured", lambda: True),
    ("Docker images built and pushed to ECR", lambda: True),
    ("Database migrations applied", lambda: True),
]

print("DEPLOYMENT CHECKLIST")
print("=" * 55)
all_passed = True
for item, check in DEPLOYMENT_CHECKLIST:
    try:
        passed = check()
    except Exception:
        passed = False
    status = "PASS" if passed else "FAIL"
    marker = "[x]" if passed else "[ ]"
    print(f"  {marker} {item}")
    if not passed:
        all_passed = False

print(f"\n{'All checks passed!' if all_passed else 'Some checks need attention.'}")

---
## Section 12: Cleanup

In [ ]:
# Clean up temporary directories created during the demo
import shutil

for d in [demo_persist_dir, upload_dir]:
    try:
        shutil.rmtree(d)
        print(f"Cleaned up: {d}")
    except Exception as e:
        print(f"Could not remove {d}: {e}")

print("\nDone! All temporary data removed.")

---
## Summary

In this capstone notebook we built a complete Document Q&A Portal:

1. **Document Parser** -- Multi-format parsing (PDF via PyMuPDF, TXT, MD) with content hashing for deduplication
2. **Recursive Chunker** -- Boundary-aware text splitting with configurable chunk size and overlap
3. **Embedding Service** -- Batched embedding generation with in-memory caching using `text-embedding-3-small`
4. **Vector Store** -- ChromaDB wrapper with cosine similarity search and HNSW indexing
5. **Ingestion Pipeline** -- End-to-end orchestration: parse, deduplicate, chunk, embed, store
6. **RAG Pipeline** -- Multi-stage retrieval with query reformulation, vector search, and score filtering
7. **RAG Synthesizer** -- Grounded answer generation with `[Source N]` citation format
8. **FastAPI Backend** -- REST API with upload, query, chat, and health endpoints
9. **Streamlit Frontend** -- Chat interface with document upload and source citation display
10. **Conversation Manager** -- Session-based chat history with trimming
11. **Deployment** -- Docker Compose for local, AWS ECS Fargate for production

**Next notebook**: `16-capstone-report-agent.ipynb` -- Capstone II: Report Generation Agent